# R6 — BERT-ER semantic blocking

R6 พยายามกู้คู่จริงที่หลุดจาก blocking เดิม โดยใช้ frozen DistilBERT → learnable hash → bucket → match head.

In [ ]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import Markdown, display

def find_root():
    candidates = [Path.cwd().resolve(), *Path.cwd().resolve().parents,
                  Path(r'D:/66070260-Year3_Term2/Project1/Code')]
    for candidate in candidates:
        if (candidate / 'exp_lib.py').exists(): return candidate
    raise FileNotFoundError('Project root containing exp_lib.py was not found')

ROOT = find_root(); EXP = ROOT / 'experiments'
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))

def source(module, *names):
    for name in names:
        display(Markdown(f'### `{module.__name__}.{name}`'))
        print(inspect.getsource(getattr(module, name)))

def read_json(relative):
    return json.loads((ROOT / relative).read_text(encoding='utf-8'))

print('Project root:', ROOT)


## 1. Architecture

DistilBERT encode profile ทีละตัว; hash head สร้าง 64-bit code สำหรับ blocking และ match head ให้ probability ของคู่ candidate.

In [ ]:
import exp_r6_bert_er as r6
source(r6, 'BertERModel', 'cosine_contrastive_loss', 'encode_all_profiles')

## 2. Train heads และ candidate recovery

Backbone ถูก freeze เพราะ fine-tune บน CPU ช้า. คู่ blocking-missed 3,316 คู่ถูก held out ไม่เข้า training.

In [ ]:
source(r6, 'build_training_pairs', 'train_heads', 'compute_hash_codes', 'build_buckets', 'blocking_recovery_eval')

## 3. แยกผล blocking ออกจาก matching

หาก hash หา candidate ไม่เจอ match head จะทำงานไม่ได้; จึงต้องดู recovery rate ควบคู่ F1.

In [ ]:
recovery=read_json('experiments/r6_blocking_recovery.json'); results=read_json('experiments/r6_results.json')
print(json.dumps(recovery, ensure_ascii=False, indent=2))
display(pd.DataFrame({'R6 manual':results['splits']['test'], 'R6 GA':results['ga_rules']['test']}).T)